# boolean

Booleans sit inside critical control flow: feature flags, permission checks, health gates, retries, validation, and deployment toggles. Many real bugs come from confusing truthiness with explicit boolean logic.


## 1. Problem First

Why does this matter in real systems?

- An empty string may incorrectly disable a feature flag.
- `0`, `None`, `[]`, and `False` can mean very different things.
- Boolean shortcuts can hide missing-data bugs.


In [ ]:
is_healthy = True
should_deploy = is_healthy and True
print(should_deploy)


## 2. Minimal Concept

Python booleans are `True` and `False`.

- Often produced by comparisons
- Used with `and`, `or`, `not`
- `bool(value)` converts a value to its truthiness


## 3. Mental Model

How Python actually behaves

`bool` is a subtype of `int`, so `True` behaves like `1` and `False` like `0` in arithmetic. Also, `and` and `or` return operands, not always literal booleans.


In [ ]:
print(isinstance(True, int))
print(True + True)
print([] or "fallback")
print("admin" and 42)


## 4. Internal Mechanics

Short-circuit evaluation stops as soon as the result is known. That affects performance, side effects, and safety.


In [ ]:
import dis

def choose(config_value, default):
    return config_value or default

dis.dis(choose)
print(choose("", "guest"))
print(choose("admin", "guest"))


## 5. Edge Cases

Where it breaks

- `0` and `None` are both falsey but semantically different.
- `value or default` can erase valid empty values.
- `and` and `or` do not guarantee `bool` output.


In [ ]:
timeout = 0
effective_timeout = timeout or 30
print(effective_timeout)

print(bool([]), bool([1]))
print(None == False)


## 6. Performance Thinking

- Boolean checks are O(1).
- Short-circuiting avoids unnecessary work.
- The bigger issue is hidden side effects in conditions.


## 7. Real Use Case

A deployment pipeline gates releases on health and approval checks. Boolean clarity matters more than cleverness.


In [ ]:
service_healthy = True
change_approved = True
traffic_below_threshold = False

can_deploy = service_healthy and change_approved and traffic_below_threshold
print(can_deploy)


## 8. Anti-Pattern

What beginners do wrong

- Use truthiness where explicit `is None` checks are required.
- Depend on `or` defaults when zero or empty string is valid.
- Write compact conditions that hide intent.


In [ ]:
user_supplied_port = 0
port = user_supplied_port or 8080
print(port)

if user_supplied_port is None:
    port = 8080
else:
    port = user_supplied_port
print(port)


## 9. Interview Signals

Questions FAANG asks

- Why is `bool` a subclass of `int`?
- What do `and` and `or` actually return?
- When should you use `is None` instead of truthiness?
- What does short-circuit evaluation buy you?


## 10. Exercise (Non-trivial)

Review a config loader that treats missing values, empty strings, and zero the same way. Redesign the logic so legitimate zero values survive while truly missing values still get defaults.


In [ ]:
def resolve_timeout(raw_timeout):
    return raw_timeout or 30

# Task:
# 1. Preserve explicit zero when it is valid.
# 2. Differentiate None from falsey-but-valid values.
# 3. Explain the truthiness bug.
